# Evaluating the output of the SCIP solver of CVXPY

In [ ]:
from av_mat_generation.CI_based.window import Window
from datetime import datetime
import pandas as pd
import numpy as np
# import cvxpy as cp
MAIN_FOLDER = 'availability_matrices/av-mat-extn-cifar10-23Feb_3'

(CVXPY) Feb 23 10:15:45 PM: Encountered unexpected exception importing solver MPAX:
RuntimeError('This version of jaxlib was built using AVX instructions, which your CPU and/or operating system do not support. You may be able work around this issue by building jaxlib from source.')


In [3]:
start_time=datetime(2022, 1, 1, 0, 0)
win = Window(start_time=start_time,random_start=False, out_folder=MAIN_FOLDER, countries=["Belgium", "Ireland", "Germany", "France", "Great Britain", "Finland", "Sweden"], n_rounds=100)
res = win.get_GHG_matrix()
min_carbon_budget = res[[i for i in range(49,50)]].sum().sum()
total_cb = res.sum().sum()
print(1-min_carbon_budget/total_cb)
win = Window(start_time=start_time,random_start=False, out_folder=MAIN_FOLDER, \
            countries=["Belgium", "Ireland", "Germany", "France", "Great Britain", "Finland", "Sweden"], n_rounds=100)

0.9922215516330158


In [4]:
carbon_budget_considered = res.sum().cumsum().to_numpy().take([39, 29, 19, 9, 4, 3, 2])
perc_cb_saved = [100*(i/total_cb) for i in carbon_budget_considered]
print(carbon_budget_considered)
print(perc_cb_saved)

[7.980065999999999 6.1960109999999995 4.235079 1.983414 0.9428550000000001
 0.7447260000000001 0.5575350000000001]
[33.03979908987285, 25.653291413710384, 17.53446140542442, 8.21191204083288, 3.903694502135956, 3.0833826959582358, 2.3083573977423577]


In [ ]:
ft=1
T = [40, 30, 20, 10, 5, 4, 3]
for CO2saving, carbon_budget in enumerate(carbon_budget_considered):
    print(T[CO2saving])
    for _in_val in [10,20,30,40,50,70,90,110,130,150]:
        win1 = Window(start_time=start_time,random_start=False, out_folder=MAIN_FOLDER, \
            countries=["Belgium", "Ireland", "Germany", "France", "Great Britain", "Finland", "Sweden"], n_rounds=T[CO2saving]+_in_val)
        win = Window(start_time=start_time,random_start=False, out_folder=MAIN_FOLDER, \
            countries=["Belgium", "Ireland", "Germany", "France", "Great Britain", "Finland", "Sweden"], n_rounds=T[CO2saving]+_in_val-ft)
        
        # Compute new budget with idle time
        C_idle = 0.1
        GHG_mat = win1.GHG_matrix.to_numpy()
        print("x.----------------------- x")
        print("Budget: ", carbon_budget)
        # display(win1.GHG_matrix)
        # print(C_idle*np.sum(GHG_mat))
        new_carbon_budget = (carbon_budget - C_idle*np.sum(GHG_mat))/(1-C_idle)
        print("New Budget: ", new_carbon_budget)
        
        if new_carbon_budget >= 0:
            try:
                av_mat = win.get_av_mat(method='greedy', fine_tuning=False, carbon_budget=new_carbon_budget-win1.get_GHG_matrix().to_numpy()[:,-ft:].sum(), key_word=f"alphaF-{T[CO2saving]+_in_val}sl-{CO2saving+1}cb")
                availability_df = pd.DataFrame(np.hstack([av_mat.to_numpy(),np.ones((7,ft))]), index=win.countries, columns=[i for i in range(av_mat.shape[1]+ft)])
                key_word=f"alphaF-{T[CO2saving]+_in_val}sl-{CO2saving+1}cb-{ft}ft"
                dict_cols = dict(
                zip([i for i in range(win.n_rounds+ft)], win1.window_list_hours[:win.n_rounds+ft])
            )  # get the datetime values
                availability_matrix_to_save = availability_df.rename(columns=dict_cols)
                availability_matrix_to_save.to_csv(
                    win.out_folder + "/av-mat_" + key_word + ".csv",
                    columns=win1.window_list_hours[:win.n_rounds+ft],
                )
            except Exception as e:
                print(f"Failed! Exception: {e}")
                continue        
    continue